![NWM](../img/NWM.png)

# Use HydroData to Retrieve Modeled and Observed Snow Data for a Watershed of Interest with ParFlow-CONUS Outputs vs Observed Snow Water Equivalent (SWE) - Full Evaluation Workflow
Authors: Irene Garousi-Nejad (igarousi@cuahsi.org), Danielle Tijerina-Kreuzer (dtijerina@cuahsi.org)  
Last updated: Feb 2026

#### Introduction:  
This notebook demonstrates how to perform a point-scale analysis comparing modeled and observed SWE at selected SNOTEL sites.  We focus on analyzing model performance both for **a single SNOTEL site** and **watershed-scale behavior for multiple stations**, with particular attention to the **magnitude and timing of peak SWE**.   

# FIX THIS: This notebook requires ParFlow-CONUS output, SNOTEL data, and metadata CSVs that are created in the `01_HydroData_collection.ipynb` notebook.

## 1. Prepare the Python Environment

Ensure that the `nwm_env` conda environment is selected as your Jupyter kernel. This environment should already be created if you followed the instructions under section "Creating your HydroLearnEnv Virtual Environment" in the `getting_started.md` file.

Import the libraries needed to run this notebook:

In [ ]:
import os
import sys
import glob
from pathlib import Path

import holoviews as hv
import hvplot.pandas
import geopandas as gpd
import pandas as pd

# Import the Evaluation library from the project root.
sys.path.append(str((Path.cwd().absolute() / "../../../src").resolve()))
from cssi_evaluation.snow import utils
from cssi_evaluation import evaluation_metrics

# import notebook-specific utilities
from utils import nwm_utils

hv.extension('bokeh')


%load_ext autoreload
%autoreload 2

# Get data from Hydrodata - from `dataCollectionHydrodata_parflow.ipynb` notebook and needs to be merged into this notebook

## 1. Setup

### 1a. Python Environment  

Ensure that the `nwm_env` conda environment is selected as your Jupyter kernel. This environment should already be created if you followed the instructions under section "Creating your HydroLearnEnv Virtual Environment" in the `getting_started.md` file.

Import the libraries needed to run this notebook:

In [ ]:
import os
import sys

prefix = os.environ['CONDA_PREFIX']
os.environ['PROJ_LIB'] = os.path.join(prefix, 'share', 'proj')

# add the src directory to the path so we can import evaluation modules
sys.path.append('../../src/')

import sys
import pyproj
import pandas as pd
import numpy as np
import xarray as xr
import geopandas as gpd
from dask.distributed import Client
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import hf_hydrodata as hf
import subsettools
import hvplot.xarray


from cssi_evaluation.utils import plot_utils


%load_ext autoreload
%autoreload 2


### 1b. Register Pin and Access HydroData

To access the HydroData catalog you will need to sign up for a [HydroFrame account](https://hydrogen.princeton.edu/signup) (do this only once), [create a 4-digit PIN](https://hydrogen.princeton.edu/pin), and register your pin in order to have access to the HydroData datasets (you will do this in the next code cell below). To note, you PIN will expire after 7 days and will need to recreate it after that time. 

In [ ]:
# You need to register on https://hydrogen.princeton.edu/pin 
# and run the following with your registered information
# before you can use the hydrodata utilities
hf.register_api_pin("dtt2@princeton.edu", "7837")

### 1c. Dask  

We'll use dask to parallelize our code. To manage parallel computation and visualize progress of long-running tasks, we initialize a Dask “cluster,” which defines how many workers are used and how much computing power each worker has. 

In this setup, we create a Dask client with `Client(n_workers=6, threads_per_worker=1, memory_limit='2GB')`, which launches a cluster with 6 workers. Each worker uses a single thread, typically mapped to one CPU core, allowing for efficient parallel processing across 6 cores. Each worker also has a memory limit of 2 GB, for a total of up to 12 GB across the cluster.


In [ ]:
# use a try accept loop so we only instantiate the client
# if it doesn't already exist.
try:
    print('Dashboard link:', client.dashboard_link)
except:    
    # The client should be customized to your workstation resources.
    client = Client(n_workers=6, threads_per_worker=1, memory_limit='2GB') 
    print('Dashboard link:', client.dashboard_link)
print(client)

## 2. Set Paths

In [ ]:
# Start and end times of a water year (to note, these dates were chosen to align with the PFCONUS1 early 2000s runs)
StartDate = '2003-10-01'
EndDate = '2005-09-30'

domain_data_path = 'examples/parflow/domain_data/' # path to the model domain data

# Path to save results (obs and mod stands for observation and modeled, respectively)
OBS_OutputFolder = './obs_outputs_PF' 
MOD_OutputFolder = './mod_outputs_PF'

## 3. Retrieve Observed Snow Data 

### 3a. Define the watershed of interest

One of the simplest ways to gather data and model output from HydroData is by specifying a [Hydrologic Unit Code](https://www.usgs.gov/national-hydrography/watershed-boundary-dataset). Before we retrieve any hydrologic information, we need to indicate a HUC8 code and use it to gather snow water equivalent (SWE) observations from SNOTEL sites  

✏️ If you have a specific HUC8 in mind, you can change the variable `huc_8_code` below.

In [ ]:
# ✏️ Specify HUC8 ID and Name for watershed of interest
huc_8_code = '14020001'  # East-Taylor HUC-8
print(f'HUC-8 ID: {huc_8_code}')

huc_8_name = 'East-Taylor'
print(f'HUC-8 name: {huc_8_name}')

Use the Subsettools function `define_huc_domain()` to get the actual CONUS1 indices associated with the East-Taylor HUC-O8. It returns a tuple `(imin, jmin, imax, jmax)` of grid indices that define a bounding box containing our region (or point) of interest (Note: (imin, jmin, imax, jmax) are the west, south, east and north boundaries of the box respectively) and a mask for that domain.

In [ ]:
ij_bounds, mask = subsettools.define_huc_domain([huc_8_code], 'conus1')

np.save(f'{domain_data_path}domainMask_{huc_8_name}_conus1.npy', mask)

plt.imshow(mask, origin='lower')
print(ij_bounds)
print(mask.shape)

Using the domain mask and the i,j PF-CONUS1 indices, we use a hf_hydrodata function to find and save the associated grid cell center lat/lon pair for each grid cell in the domain. 

In [ ]:
# Extract bounds
i_min, j_min, i_max, j_max = ij_bounds
mask_shape = mask.shape #shape of the subset rectangular domain

# Create i/j index ranges
i_vals = np.arange(i_min, i_max)
j_vals = np.arange(j_min, j_max)

# Create full 2D grid (note indexing order carefully)
jj, ii = np.meshgrid(j_vals, i_vals, indexing="ij")

Because the function `hf.to_latlon()` finds the coordinates at the lower left corner of a grid cell, we add 0.5 to each i,j index pair to find the **lat/lon at the grid cell center**.

In [ ]:
# Compute grid cell centers
ii_center = ii + 0.5
jj_center = jj + 0.5

# Convert to lat/lon (vectorized loop)
lat = np.zeros(mask_shape)
lon = np.zeros(mask_shape)

for r in range(mask_shape[0]):
    for c in range(mask_shape[1]):
        lat[r, c], lon[r, c] = hf.to_latlon("conus1",
                                            ii_center[r, c],
                                            jj_center[r, c])

In [ ]:
# Save 2D arrays of Lat & Lon
np.save(f"{domain_data_path}{huc_8_name}_{huc_8_code}_lat_2d.npy", lat)
np.save(f"{domain_data_path}{huc_8_name}_{huc_8_code}_lon_2d.npy", lon)

# Save the Lat & Lon in a df (as CSV) matched with the ParFlow-CONUS i,j indices 
grid_df = pd.DataFrame({
    "i": ii.ravel(),
    "j": jj.ravel(),
    "lat": lat.ravel(),
    "lon": lon.ravel(),
})
grid_df.to_csv(f"{domain_data_path}df_{huc_8_name}_{huc_8_code}_conus1_gridPoints_LatLon.csv", index=False)

# Save a shapefile of the watershed Lat & Lon
grid_gdf = gpd.GeoDataFrame(
    grid_df,
    geometry=gpd.points_from_xy(grid_df.lon, grid_df.lat),
    crs="EPSG:4326"
)
# Save the grid points / GeoDataFrame to a shapefile for later use
grid_gdf.to_file(f"{domain_data_path}{huc_8_name}_{huc_8_code}_conus1_gridPoints_LatLon.shp")

In [ ]:
grid_df

### 3b. Explore the available SWE data in a watershed  

<div style="color:black;background-color:#f5f5f5; padding:10px; border-left: 5px solid #007acc;">
<h4>📖 Did you know?</h4>
<p>The Snow Telemetry (SNOTEL) network, managed by the USDA Natural Resources Conservation Service (NRCS), monitors snowpack conditions across key watersheds in the western United States to support water supply forecasting and climate monitoring. SNOTEL sites are fully automated stations that continuously measure snow water equivalent (SWE), snow depth, precipitation, temperature, and other meteorological variables throughout the year. Unlike manual snow survey programs, SNOTEL provides high-temporal-resolution observations that enable near–real-time assessment of snowpack evolution and interannual variability. These data are widely used for operational forecasting, drought assessment, and long-term climate analysis. </p>
</div>

Explore what SWE data is available at sites within the HUC ID you specified that operated during WY2004 and WY2005. If you want to check other variables besides SWE, you can change the `variable` argument name. 

In [ ]:
avail_df = hf.get_site_variables(variable = "swe",
                        huc_id = [huc_8_code], grid = 'conus1',
                        date_start = StartDate, date_end = EndDate)

# View first five records
avail_df.head(5)

### 3c. Map the SNOTEL stations inside the HUC-08 watershed that have available data in the selected time range  
To note here, we are using pre-loaded shape files for the East-Taylor HUC8, which are located in the `/domain_data/` directory.

In [ ]:
### Select station locations that fall within the HUC8 watershed

# Path to the watershed shapefile that was just created
watershed = f'{domain_data_path}East-Taylor_14020001.shp'
watershed_gdf = gpd.read_file(watershed).to_crs(epsg=4326)

# Create GeoDataFrame of all available stations
filtered_all_stations_gdf = gpd.GeoDataFrame(
    avail_df,
    geometry=gpd.points_from_xy(
        avail_df.longitude,
        avail_df.latitude
    ),
    crs="EPSG:4326"
)
print("Sites CRS:", filtered_all_stations_gdf.crs)

# Combine watershed polygons into one geometry
watershed_union = watershed_gdf.geometry.unary_union

# Filter stations that fall within the watershed
sites_in_watershed = filtered_all_stations_gdf[
    filtered_all_stations_gdf.geometry.within(watershed_union)
].copy()

sites_in_watershed.reset_index(drop=True, inplace=True)

print(f"Total sites in watershed: {len(sites_in_watershed)}")




Plot these sites on a map. Then, hover over the pins to see the site names.

In [ ]:
## TODO: REPLACE WITH CSSI_EVALUATION.PLOTS FUNCTIONS

# this may take a moment to load, but it should pop up in a new window
m = plot_utils.plot_sites_within_domain(sites_in_watershed, watershed_gdf, zoom_start=9)
m

## 4. Retrieve SNOTEL point observations and metadata from HydroData  
Use the `hf.get_point_data()` function to retrieve daily, start-of-day SWE from SNOTEL sites:

In [ ]:
# Create a folder to save observations
isExist = os.path.exists(OBS_OutputFolder)
if isExist == True:
    exit
else:
    os.mkdir(OBS_OutputFolder)

### 4a. Get HydroData Observed SWE
Gather the SNOTEL data for all stations within the watershed and save CSV:

In [ ]:
# Request point observations data
data_df = hf.get_point_data(dataset="snotel", variable="swe", temporal_resolution="daily", aggregation="sod",
                         date_start=StartDate, date_end=EndDate,
                         huc_id=[huc_8_code], grid='conus1')
                         #polygon=watershed_bbox, polygon_crs=watershed_crs)

# save
data_df.to_csv(f'./{OBS_OutputFolder}/df_{huc_8_name}_{huc_8_code}_SNOTEL.csv', index=False)

# Ensure date column is datetime
data_df["date"] = pd.to_datetime(data_df["date"])

# View first five records
data_df.head(5)

### 4b. Get Metadata for HydroData Observed SWE
Also, retrieve the metadata for the same stations we retrieved SWE observations for:

In [ ]:
# Request site-level attributes for these sites
metadata_df = hf.get_point_metadata(dataset="snotel", variable="swe", temporal_resolution="daily", aggregation="sod",
                                 date_start=StartDate, date_end=EndDate,
                                 huc_id=['14020001'], grid='conus1')

# save
metadata_df.to_csv(f'./{OBS_OutputFolder}/df_{huc_8_name}_{huc_8_code}_SNOTEL_metadata.csv', index=False)

# View first five records
metadata_df.head(5)

The metadata file is an important addition to the observations and it is recommended to always gather and save this for the observations you are using (particularly to support reproducibility within an open-science workflow). The saved file has useful attributes like site names, first and last date of available data, lat/lon, and the query URL.  

Additionally, the metadata contains **ParFlow-CONUS1 and ParFlow-CONUS2 `i,j` indices, which indicate the exact model domain grid cell the observation aligns with**. This is a useful HydroData feature that removes the need for users to manually match station latitude/longitude coordinates to the appropriate model grid cell, as this spatial mapping is handled directly within HydroData. We will use these indices below to extract PF-CONUS1 modeled SWE for each SNOTEL station in the section below. 

## 5. Retrieve ParFlow-CONUS1 Modeled Snow Data

In [ ]:
# Create a folder to save results
isExist = os.path.exists(MOD_OutputFolder)
if isExist == True:
    exit
else:
    os.mkdir(MOD_OutputFolder)

The following section retrieves ParFlow-CONUS1 data for each SNOTEL site within our HUC-08 watershed. The code identifies the CONUS1 `i,j` indices associated with each SNOTEL site, indicated in the `metadata_df`. It then extracts the CONUS1 modeled SWE output for the site and the period of interest, returning the result as a DataFrame. To fairly compare with SNOTEL, which reports SWE once daily at the start of the local day, model output is aggregated by day, using the argment `"temporal_resolution": "daily"`. Finally, the processed data is saved as a CSV file for each site.  

### 5a. ParFlow CONUS1 Model Dataset Information
We can print some information about the model output dataset by using the `hf.get_catalog_entry()` to get the CONUS1 model dataset metadata. 

In [ ]:
conus1_options = {
    "dataset": "conus1_baseline_mod",
    "variable": "swe"
}
hf.get_catalog_entry(conus1_options)

Before we gather model outputs at the specific SNOTEL sites, we can visualize SWE across our HUC-08. This is plotted for one day at 1km lateral resolution.

In [ ]:
# retrieve gridded PF-CONUS1 SWE for the entire HUC8 watershed
grid_swe_options = {
        "dataset": "conus1_baseline_mod",
        "variable": "swe",
        "temporal_resolution": "daily",
        "start_time": '2004-04-01', ### TO NOTE: the gridded function has exclusive end date, so this is hardcoded for now 
        "end_time": '2004-04-02',
        "huc_id": huc_8_code
    }
    
    # Get gridded data
grid_swe = hf.get_gridded_data(grid_swe_options)

In [ ]:
grid_swe_map = xr.DataArray(grid_swe[0], dims=("y", "x"), name="SWE")
grid_swe_map.hvplot.image(cmap="YlGnBu", colorbar=True, aspect="equal", title=f"{huc_8_name} Gridded SWE on 2004-04-01")


Now, grab the PF-CONUS1 modeled SWE from the SNOTEL site locations. Here we use the CONUS1 i and j indices from the `metadata_df` and grab the SWE from those grid cells.  

In [ ]:
# Copy data_df to model_df so we have the same timestamps and site_id structure
model_df = data_df.copy()

# Set all non-date columns to NaN to prepare for filling in model data
non_date_cols = model_df.columns.difference(["date"])
model_df[non_date_cols] = np.nan

# Rename site_id columns for PF outputs 
model_df.columns = [
    col if col == "date" else col.replace(":SNTL", "") + ":PFCONUS1"
    for col in model_df.columns
]

Use the function `hf.get_gridded_data()` and PF-CONUS1 `i,j` indices to select the SWE output for the correct location and time period: 

In [ ]:
# Loop over each station in metadata_df
for idx, row in metadata_df.iterrows():
    site_id = row["site_id"]  # original SNTL site_id
    col_name = site_id.replace(":SNTL", "") + ":PFCONUS1"  # corresponding column in model_df
    conus_i = int(row["conus1_i"])
    conus_j = int(row["conus1_j"])
    
    # Build options dict for this station
    options = {
        "dataset": "conus1_baseline_mod",
        "variable": "swe",
        "temporal_resolution": "daily",
        "start_time": '2003-10-01', ### TO NOTE: the gridded function has exclusive end date, so this is hardcoded for now 
        "end_time": '2005-10-01',
        "grid_point": [conus_i, conus_j]
    }
    
    # Get gridded data
    data = hf.get_gridded_data(options)
    
    # Fill column in model_df
    # Convert to numeric in case hf returns lists or other types
    model_df[col_name] = np.squeeze(np.array(data))

# Ensure date column is datetime
model_df["date"] = pd.to_datetime(model_df["date"])

# Save
model_df.to_csv(f'./{MOD_OutputFolder}/df_{huc_8_name}_{huc_8_code}_PFCONUS1.csv', index=False)
    
model_df.head(5)

## 6. Quick plot sanity check  
Plot a simple timeseries of modeled and observed SWE to make sure our data retrieval was successful. 

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(data_df["date"], model_df["380:CO:PFCONUS1"], label="Modeled", linewidth=2)

ax.plot(data_df["date"], data_df["380:CO:SNTL"], label="Observed", linewidth=2)

ax.set_xlabel("Date")
ax.set_ylabel("SWE (mm)")

# Date formatting for x-axis
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%Y'))

ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()

# Start of comparison from old notebook

## 3. Spatial Mapping of the SNOTEL sites  
Before evaluating model performance, we plot the GIS data associated with the records in the combined DataFrame. The map below shows the SNOTEL stations included in the evaluation dataset, along with the watershed boundary used for the model simulations. Hover over the pins to see the site names.  

We also print a table of the SNOTEL site metadata to help with the single site selection.

In [ ]:
# Path to the watershed shapefile
watershed = "./domain_data/East-Taylor_14020001.shp"
watershed_gdf = gpd.read_file(watershed).to_crs(epsg=4326)

# Create GeoDataFrame of all available stations
filtered_all_stations_gdf = gpd.GeoDataFrame(
    metadata_df,
    geometry=gpd.points_from_xy(
        metadata_df.longitude,
        metadata_df.latitude
    ),
    crs="EPSG:4326"
)
print("Sites CRS:", filtered_all_stations_gdf.crs)

# Combine watershed polygons into one geometry
watershed_union = watershed_gdf.geometry.unary_union

# Filter stations that fall within the watershed
sites_in_watershed = filtered_all_stations_gdf[
    filtered_all_stations_gdf.geometry.within(watershed_union)
].copy()

sites_in_watershed.reset_index(drop=True, inplace=True)

print(f"Total sites in watershed: {len(sites_in_watershed)}")

m = nwm_utils.plot_sites_within_domain(sites_in_watershed, watershed_gdf, zoom_start=9)
m

In [ ]:
sites_in_watershed

## 4. Compare Modeled and Observed SWE Timeseries at a Single Site

Once we have both observation data and modeling outpus, it's important to evaluate how well the model reproduces observed data. The following plots are simple timeseries comparisons of **modeled vs. observed** SWE. These types of plots provide a straight-forward visual of how well the observations and simulations agree and are a great start for assessing general model performance.  

📊 We include two figures:

1. **Time Series Overlay:** Plots the observed and modeled values together over time. This helps identify:
   - Periods of systematic bias
   - Timing differences in peaks and lows
   - General agreement in trends

2. **Scatter Plot with 1:1 Line:** Plots each modeled value against its corresponding observed value. This highlights:
   - Accuracy across the full range of values
   - Over- or under-prediction patterns
   - Outliers or extreme events

Review the sites within the watershed from the interactive map above and click on the markers to view the site name and code. Recall, we also printed out the site metadata for all sites within the watershed, which contains the 3-letter site codes.

✏️ Once you’ve identified the site of interest, **enter its site code in the next code cell for `my_site_code`**:   

In [ ]:
# choose a site of interest within the watershed
my_site_code = '380:CO:'

############################ THIS BELOW DOESNT WORK BECAUSE CODE IS NOT COMPLETE
# filter to only that site
sites_in_watershed[sites_in_watershed['site_id']==my_site_code]

In [ ]:
nwm_utils.comparison_plots(combined_df, f'{my_site_code}SNTL', f'{my_site_code}PFCONUS1')

To move beyond an overall summary of daily performance, we replot the modeled vs. observed SWE scatter while highlighting specific months with a distinct color. This gives us more information about the **seasonal model performance**.  

Let's customize the scatter plot by allowing you to highlight specific months with a distinct color. The selected months will appear in one color, while all other months will appear in a different color. This customization reveals whether there are **seasonal patterns** in the relationship between observed and modeled SWE, allowing us to distinguish model behavior during the key snowpack phases of <span style="color: teal;">accumulation</span> and <span style="color: tomato;">ablation (melt)</span>. Identifying these patterns is important for diagnosing the model’s strengths and limitations during different parts of the snow season.

You can change the list of highlighted months (for example, October–December for early accumulation or March–May for spring melt) to explore in the scatter plot how model performance varies across different parts of the snow season. This seasonal perspective motivates the _peak SWE analysis_ that follows.

##### 📊 For this example, let's highlight the _early snow accumulation period_ of October - January:

In [ ]:
combined_df['month'] = combined_df.index.month

plot = nwm_utils.plot_custom_scatter_SWE(combined_df, f'{my_site_code}SNTL', f'{my_site_code}PFCONUS1',
                                        highlight_months=[10, 11, 12, 1])
plot 

<div style="color:black; background-color:#f5f5f5; padding:10px; border-left: 5px solid #007acc;">
<h4>🧠 Reflect</h4>
<p>What does this plot tell us about how well the model performs during the early snow accumulation period at this site?<br>
HINT: How close are the <span style="color: teal;">green points</span> to the 1:1 line?</p>
</div>

## 5. Peak SWE Evaluation at the Watershed Scale  
As we saw in the previous section, how well a model matches observations can differ greatly throughout the year. The following section focuses on **peak SWE** (or maximum SWE) analysis. 

**Peak SWE is a key diagnostic for snow-dominated hydrologic systems** because it represents the maximum amount of liquid water stored in the snowpack before the spring melt. Evaluating both the magnitude (quantity) and timing (date) of peak SWE provides insight into whether the model is accurately representing snow accumulation and seasonal energy balance.  

Errors in peak SWE can have important hydrologic consequences, as peak accumulation strongly influences:
- The volume of water available for spring runoff
- The timing of streamflow peaks
- Soil moisture recharge and groundwater contributions

<div style="padding: 10px;">
  <img src="img/peak_swe_example.png" style="width:85%;">
</div>  

_Example daily SWE at a single site, showing two important periods in snow processes: accumulation (before peak) and ablation (after peak). The vertical line marks peak SWE._

### 5.1 Comparing Modeled and Observed Peak SWE at All Sites in the Watershed
In this section, we evaluate observed and modeled peak SWE for all stations within our watershed and for all years selected in the `StartDate` and `EndDate` above. 

#### 📋 Modeled SWE on the Date of Observed Peak SWE (magnitude)  
This comparison evaluates the modeled SWE on the **specific day when observed SWE reaches its maximum.** By fixing the timing to the observed peak, this comparison isolates errors in SWE magnitude.  
It answers the question: *How much SWE does the model simulate on the day the observed snowpack reaches its maximum?*

In [ ]:
# isolate the columns associated with observations and model predictions.
# these will be inputs to our same-day comparison function.
obs_cols = sorted([col for col in combined_df.columns if col.endswith('SNTL')])
mod_cols = sorted([col for col in combined_df.columns if col.endswith('PFCONUS1')])

In [ ]:
# compute the same-day SWE comparison during the observed peak SWE for each of the observation and modeled sites.
df_observed_peak = utils.modeled_swe_at_observed_peak(combined_df, obs_cols, mod_cols)
df_observed_peak

##### 📊 Visualize the amount of SWE on **the day of observed peak SWE occurs** for both the model and observations at each station

In [ ]:
# Rearrange the dataframe to long format for easier plotting
df_long = (
    df_observed_peak
    .reset_index()  
    .melt(
        id_vars=['Station', 'Water_Year', 'date'],
        value_vars=['Observed', 'Modeled'],
        var_name='Source',
        value_name='SWE'
    )
)
# Create scatter plot of observed and modeled SWE on the day of observed peak SWE
scatter_obs_peak = df_long.hvplot.scatter(
    x='Station',
    y='SWE',
    by='Source',  # Observed vs Modeled
    ylabel='SWE on Observed Peak Day (mm)',
    title='Observed and Modeled SWE on the Day of Observed Peak SWE',
    size=70,
    width=700,
    height=450,
    alpha=0.8,
    hover_cols=['Water_Year'],
    rot=45
)

# Customize the scatter plot appearance
scatter_by_station = (
    scatter_obs_peak  
    .opts(legend_position='top_right')
)

scatter_by_station

#### 📋 Modeled vs Observed Peak SWE Comparison (timing & magnitude)  
This comparison evaluates the modeled and observed peak SWE values and their corresponding dates independently. Unlike the previous comparison that fixed the timing to the observed peak swe, this analysis shows the actual days of modeled and observed peak SWE, which may occur on different dates. As a result, it captures errors in both **peak SWE magnitude** and **peak timing**.

In [ ]:
# compute the different-day SWE comparison for each of the observed and modeled sites.
df_both_peak = utils.modeled_vs_observed_peak_swe(combined_df, obs_cols, mod_cols)
df_both_peak

##### 📊 Visualize the quantity of peak SWE for both the model and observations at each station

In [ ]:
### NEED TO DECIDE HOW TO FORMAT THIS PLOT AND IF WE WANT TO HAVE THE "SAME_DAY" PLOT

# Create the scatter plot
scatter_plot_both_peak = df_both_peak.hvplot.scatter(
    x='Observed',
    y='Modeled',
    xlabel='Observed SWE (mm)',
    ylabel='Modeled SWE (mm)',
    title='Modeled vs. Observed Peak SWE',
    size=35,
    width=500,
    height=400,
    color='#E69F00',
    hover_cols=['Station', 'Water_Year']
)#.relabel('Peak SWE')

# Add 1:1 line (perfect match line)
swe_max = df_both_peak[['Observed', 'Modeled']].max().max()

one_to_one_line = hv.Curve(([0, swe_max], [0, swe_max])).opts(
    color='gray',
    line_dash='dashed',
    line_width=1,
).relabel('1:1 Line')

# Combine scatter plot and 1:1 line into an Overlay
scatter_with_line = (scatter_plot_both_peak * one_to_one_line).opts( #scatter_plot_obs_peak * 
    legend_position='bottom_right'
)

scatter_with_line

### 5.2 Visualizing Model Error for Peak SWE

The previous scatter plots indicate that the modeled and observed peak SWE magnitude and timing don't always align. Next, we plot the degree to which   

The previous scatter plots highlight differences between modeled and observed peak SWE timing and magnitude, but interpreting these variations can be challenging when comparing modeled and observed values directly. To make these differences more explicit, we compute errors in both peak timing and peak SWE magnitude and visualize them directly. This approach clarifies both the direction and magnitude of model bias and facilitates comparison across stations and water years.

First, add columns `Peak_Date_Diff_Days` and `Peak_SWE_Diff` to the DataFrame `df_both_peak` for computed difference in peak SWE date difference and peak SWE quantity between modeled and observed:

In [ ]:
# Compute the difference in peak SWE days and peak SWE amounts between modeled and observed
df_both_peak['Peak_Date_Diff_Days'] = (df_both_peak['Modeled_Date'] - 
                                      df_both_peak['Observed_Date']).dt.days
df_both_peak['Peak_SWE_Diff'] = (df_both_peak['Modeled'] - 
                           df_both_peak['Observed'])

In [ ]:
df_both_peak

##### 📊 Visualize the error between the modeled and observed peak SWE  

In [ ]:
# Filter to separate each water year
year1 = df_both_peak[df_both_peak['Water_Year'] == df_both_peak['Water_Year'].min()]
year2 = df_both_peak[df_both_peak['Water_Year'] == df_both_peak['Water_Year'].max()]

bar1 = year1.hvplot.bar(
    x='Station',
    y='Peak_Date_Diff_Days',
    rot=45,
    ylabel='Date Difference (days)',
    title=f'Peak SWE Date Difference {year1["Water_Year"].iloc[0]} (model - obs)',
    width=400,
    height=400,
    color='Peak_Date_Diff_Days',
    hover_cols=['Modeled', 'Observed']
)
bar2 = year1.hvplot.bar(
    x='Station',
    y='Peak_SWE_Diff',
    rot=45,
    ylabel='SWE Difference (m)',
    title=f'Peak SWE Difference {year1["Water_Year"].iloc[0]} (model - obs)',
    width=400,
    height=400,
    color='Peak_SWE_Diff',
    hover_cols=['Modeled', 'Observed']
)

# Combine side by side
layout = (bar1 + bar2)
layout.opts(shared_axes=False)

The left panel shows the timing error (date difference) and the right panel the magnitude error (SWE difference). When we computed the difference in date and SWE quantity above, we took `modeled - observed` so:  

| | DATE OF PEAK SWE | PEAK SWE QUANTITY |
|---|---|---|
| + Positive Values | modeled AFTER observed | modeled GREATER THAN observed |
| - Negative Values | modeled BEFORE observed | modeled LESS THAN observed |  

<div style="color:black; background-color:#f5f5f5; padding:10px; border-left: 5px solid #007acc;">
<h4>🧠 Reflect</h4>
<p>Looking at the two plots, what could be some reasons for the model having simulated peak SWE both earlier and less than the observed peak SWE? Perhaps try changing the <code>my_site_code</code> from earlier in the notebook to rerun <code>nwm_utils.comparison_plots()</code> to see the timeseries for a different station to look at the peak magnitude and timing. 

<br>What happens if you change the year that is plotted? <br>✏️ Try modifying the bar plot code from <code>bar1 = year1.hvplot.bar</code> to <code>bar1 = year2.hvplot.bar</code>. Don't forget to change the title!</p>
</div>

#### 📊 Next, we combine the timing and magnitude errors and plot them together for each station.

In [ ]:


scatter = df_both_peak.hvplot.scatter(
    x='Peak_Date_Diff_Days',
    y='Peak_SWE_Diff',
    by='Station', # Water_Year
    xlabel='Peak SWE Timing Error (days)',
    ylabel='Peak SWE Magnitude Error (mm)',
    title='Peak SWE Timing vs Magnitude Error',
    size=80,
    width=600,
    height=400,
    hover_cols=['Water_Year']
)

# Add reference lines
vline = hv.VLine(0).opts(color='gray', line_dash='dashed')
hline = hv.HLine(0).opts(color='gray', line_dash='dashed')

(scatter * vline * hline).opts(legend_position='top_left', show_grid=True)


✏️ **Try changing how we view this plot.**  
We can modify a line in the section of code from `by='Station'` to `by='Water_Year'` to better visualize the errors in the different Water Years.  
Are there any patterns that jump out? Which year was modeled peak SWE consistently less than observed peak SWE? 

## 6. Compute and Statistics and Error Metrics  
The previous section visualized when and where modeled SWE differs from observations, both in terms of peak SWE timing and magnitude. However, visual inspection alone makes it difficult to compare performance across sites or to summarize model behavior in a consistent or quantifiable way. In this section, we compute commonly used statistical error metrics to quantify model performance, allowing us to objectively assess bias, error magnitude, and variability for sites within the watershed.  

Proposed outline (DTK, Jan 2026):
- Summary metrics at a station
- Summary metrics at all stations within the watershed
- Combined timing and magnitude for all stations within the watershed (Condon metric)
- Focus on timing: summary statistics for single station for accumulation & ablation periods (using the new wrapper: `nwm_utils.compute_stats_period()`)
- Melt period statistics

In [ ]:
nwm_utils.compute_stats(combined_df, f'CCSS_{my_site_code}_swe_m', f'NWM_{my_site_code}_swe_m')

Pearson and Spearman correlations are both close to 1, suggesting a strong relationship between observed and modeled SWE. As shown on the timeseries plot, this strong correlation alone does not indicate a "good" model. For example, it does not guarantee accurate timing of key events, such as peak SWE or melt onset. Let's compare these as well. The following code uses `report_max_dates_and_values` function to identify the peak SWE value and the date it occurs for both the observed (CCSS) and modeled (NWM) datasets. 

<div style="color:black; background-color:#f5f5f5; padding:10px; border-left: 5px solid #007acc;">
<h4>🧠 Reflect</h4>
<p>You now have several performance metrics: Bias, Pearson Correlation, Spearman Correlation, NSE, and KGE. If you had to pick just one metric to summarize model performance, which would you choose—and why? As you review the results, compare the peak flow amounts and the timing of snowmelt onset. Do you see any significant differences? Which dataset indicates an earlier melt?</p>
</div>

In [ ]:
summary_table = nwm_utils.report_max_dates_and_values(combined_df, f'CCSS_{my_site_code}_swe_m', f'NWM_{my_site_code}_swe_m')
summary_table

### Summary Metrics at Multiple Sites

In [ ]:
site_codes = ['DAN', 'HRS', 'KIB', 'PDS', 'SLI', 'TUM', 'WHW']

rows = []

for site in site_codes:
    obs_col = f'CCSS_{site}_swe_m'
    mod_col = f'NWM_{site}_swe_m'

    stats_table = nwm_utils.compute_stats(combined_df, obs_col, mod_col)

    rows.append({
        'Station': site,
        'Mean_Obs': stats_table.loc['observed', 'Mean'],
        'Mean_Mod': stats_table.loc['modeled', 'Mean'],
        'Bias_m': stats_table.loc['Bias (Modeled - Observed)', 'Mean'],
        'Pearson_r': stats_table.loc['Pearson Correlation', 'Mean'],
        'Spearman_r': stats_table.loc['Spearman Correlation', 'Mean'],
        'NSE': stats_table.loc['Nash-Sutcliffe Efficiency (NSE)', 'Mean'],
        'KGE': stats_table.loc['Kling-Gupta Efficiency (KGE)', 'Mean']
    })

stats_AllStations = pd.DataFrame(rows)

print('All Stations Statistics Summary:')
stats_AllStations

In [ ]:
stats_AllStations.hvplot.bar(
    x='Station',
    y='NSE',
    rot=45,
    ylabel='Nash–Sutcliffe Efficiency',
    title='NSE by Station',
    height=400,
    width=600,
    bar_width=0.5
)


In [ ]:
stats_summary.hvplot.scatter(
    x='Station',
    y='Bias_m',
    size=100,
    rot=45,
    ylabel='Bias (m)',
    title='Mean SWE Bias by Station'
)


### Combine Magnitude (absolute relative bias) and Timing (Spearman's rho) metrics using the Condon metric (and with all stations, a Condon diagram)

In [ ]:
bias1 = evaluation_metrics.bias(combined_df.CCSS_TUM_swe_m, combined_df.NWM_TUM_swe_m)
bias1

In [ ]:
abs_bias = evaluation_metrics.absolute_relative_bias(combined_df.CCSS_TUM_swe_m, combined_df.NWM_TUM_swe_m)
abs_bias

In [ ]:
srho = evaluation_metrics.spearman_rank(combined_df.CCSS_TUM_swe_m, combined_df.NWM_TUM_swe_m)
srho

In [ ]:
evaluation_metrics.condon(abs_bias, srho)

<div style="color:black; background-color:#f5f5f5; padding:10px; border-left: 5px solid #007acc;">
  <h4>🧠 Reflect</h4>
  <p>
    What is the modeled SWE on the date when the observed SWE reaches its peak?<br>
    ✏️ Use the code snippet below to find the answer.
  </p>
  <pre><code class="language-python">
  
    # Find date of the peak SWE from observed data
    date_obs_max = combined_df['CCSS_HRS_swe_m'].idxmax()

    # Get corresponding value of modeled data on that date
    value_mod_at_max_obs = combined_df.loc[date_obs_max, 'NWM_HRS_swe_m']
  </code></pre>
</div>


### Focus on Timing: Melt Period Metrics
Compare the average melt rate over the full melt period. 

The following function computes the melt period length by identifying the first date after the peak SWE when SWE drops to zero and remains at zero for at least (`min_zero_days`) consecutive days. This is used to define the end of the melt period. Finally, the function calculates the average melt rate, which represents the rate at which snow disappeared, expressed in meters per day, over the full melt period.

In [ ]:
melt_stats_df = utils.compute_melt_period_statistics(combined_df)
melt_stats_df.head()
melt_stats_df

In [ ]:
observed_melt_period = nwm_utils.compute_melt_period(combined_df[f'CCSS_{my_site_code}_swe_m'])
observed_melt_period

In [ ]:
modeled_melt_period = nwm_utils.compute_melt_period(combined_df[f'NWM_{my_site_code}_swe_m'])
modeled_melt_period

In [ ]:
accum_months = [10, 11, 12, 1, 2, 3]
ablation_months = [4, 5, 6]

accum_stats = nwm_utils.compute_stats_period(
    combined_df,
    f'CCSS_{my_site_code}_swe_m',
    f'NWM_{my_site_code}_swe_m',
    accum_months
)

accum_stats

In [ ]:

ablation_stats = nwm_utils.compute_stats_period(
    combined_df,
    f'CCSS_{my_site_code}_swe_m',
    f'NWM_{my_site_code}_swe_m',
    ablation_months
)

ablation_stats

<div style="color:black; background-color:#f5f5f5; padding:10px; border-left: 5px solid #007acc;">
  <h4>🧠 Reflect</h4>
  <p>
    If you recall from earlier, we plotted the timeseries of out selected station. Replot it below. Do the metrics make sense given the visual comparison between modeled and observed? For example, when you look at the timeseries, is the model consistently predicting SWE to be higher or lower than observations? Does this align with the <b>Bias</b> sign (+ or -)?
  </p>
</div> 

In [ ]:
nwm_utils.comparison_plots(combined_df, f'CCSS_{my_site_code}_swe_m', f'NWM_{my_site_code}_swe_m')
